# BITMAN Info Map

## Assignment 4: Scraping and Unstructured Data

In this notebook, I am scraping course information from the BCIT website and turning it into a structured dataset. I am focusing only on the common first-year BITMAN courses and the ADM option not **including ESM & AIM** so the project stays clear and manageable. *ill be honest im starting this late and i dont have much time either with exams coming up*.

My goal is to take messy webpage text and turn it into something easier to read, organize, and use.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

## Choosing the pages

Here, I am using the two main BCIT pages I need for this project. One page is for the main BITMAN diploma, and the other is for the ADM option.

I will request both pages and make sure they load properly before I start scraping.

In [2]:
main_url = "https://www.bcit.ca/programs/business-information-technology-management-diploma-full-time-6235dipma/"
adm_url = "https://www.bcit.ca/programs/business-information-technology-management-analytics-data-management-option-diploma-full-time-623cdipma/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

main_response = requests.get(main_url, headers=headers)
adm_response = requests.get(adm_url, headers=headers)

print("Main page status:", main_response.status_code)
print("ADM page status:", adm_response.status_code)

Main page status: 200
,ADM page status: 200


In [3]:
main_soup = BeautifulSoup(main_response.text, "html.parser")
adm_soup = BeautifulSoup(adm_response.text, "html.parser")

print(main_soup.title.get_text(strip=True))
print(adm_soup.title.get_text(strip=True))

Business Information Technology Management, Diploma, Full-time – BCIT
,Business Information Technology Management (Analytics Data Management Option), Diploma, Full-time – BCIT


## Making a function to find courses

The BCIT pages have a lot of text, so I made a function to help me find the lines that look like course entries. *I tried to do soemthing but completely fumbled, it didnt work so now deleting all of that and redoing doing something different* 

I want the function to catch the course code and course name so I can start building my dataset.

### Note: My earlier method did not work well because it tried to read the whole page as plain text 

In [4]:
main_links = []

for a in main_soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]
    
    if text:
        main_links.append((text, href))

for item in main_links[:80]:
    print(item)

('Skip to main content', '#a11y-skip-link-content')
,('Learning Hub', 'https://learn.bcit.ca/')
,('myBCIT', 'https://my.bcit.ca/')
,('Students', '/current-students/')
,('Staff', '/faculty-staff/')
,('Events', '/events/')
,('Programs & Courses', '/study/')
,('Applied & Natural Sciences', '/study/applied-natural-sciences/')
,('Business & Media', '/study/business-media/')
,('Computing & IT', '/study/computing-it/')
,('Engineering', '/study/engineering/')
,('Health Sciences', '/study/health-sciences/')
,('Trades & Apprenticeships', '/study/trades-apprenticeships/')
,('Flexible Learning', 'https://www.bcit.ca/flexible-learning/')
,('International', '/international-applicants/')
,('Academic Dates & Deadlines', '/academic-dates/')
,('Admission', '/admission/')
,('Future Students', 'https://www.bcit.ca/admission/future-students/')
,('How to Apply', 'https://www.bcit.ca/admission/how-to-apply/')
,('Entrance Requirements', 'https://www.bcit.ca/admission/entrance-requirements/')
,('Tuition & Fees

In [5]:
for text, href in main_links:
    if "/courses/" in href or "/outlines/" in href:
        print(text, "->", href)

OPMT 0199 -> /courses/math-for-business-opmt-0199/
,OPMT 0198 -> /courses/business-math-assessment-test-opmt-0198/
,OPMT 0199 -> https://www.bcit.ca/courses/math-for-business-opmt-0199/
,OPMT 0023 -> https://www.bcit.ca/courses/business-math-refresher-opmt-0023/
,course outline -> /outlines/bsys1000/
,course outline -> /outlines/comm1100/
,course outline -> /outlines/econ2100/
,course outline -> /outlines/mktg1102/
,course outline -> /outlines/opmt1103/
,course outline -> /outlines/opmt1110/
,course outline -> /outlines/bsys2000/
,course outline -> /outlines/bsys2065/
,course outline -> /outlines/busa2150/
,course outline -> /outlines/comm2200/
,course outline -> /outlines/econ2200/
,course outline -> /outlines/fmgt2152/
,course outline -> /outlines/opmt1130/
,course outline -> /outlines/opmt1174/


In [6]:
adm_links = []

for a in adm_soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    href = a["href"]
    
    if text:
        adm_links.append((text, href))

for text, href in adm_links:
    if "/courses/" in href or "/outlines/" in href:
        print(text, "->", href)

course outline -> /outlines/babi3000/
,course outline -> /outlines/babi3005/
,course outline -> /outlines/bsys3000/
,course outline -> /outlines/bsys3205/
,course outline -> /outlines/bsys3355/
,course outline -> /outlines/busa4850/
,course outline -> /outlines/fmgt3221/
,course outline -> /outlines/opmt3301/
,course outline -> /outlines/babi4000/
,course outline -> /outlines/babi4005/
,course outline -> /outlines/blaw3600/
,course outline -> /outlines/bsys4000/
,course outline -> /outlines/bsys4905/
,course outline -> /outlines/busa4800/
,course outline -> /outlines/fmgt4530/
,course outline -> /outlines/opmt4170/


In [9]:
from urllib.parse import urljoin
import time

In [10]:
def get_outline_links(soup, base_url, stream_name):
    rows = []
    
    for a in soup.find_all("a", href=True):
        href = a["href"]
        
        if "/outlines/" in href:
            full_url = urljoin(base_url, href)
            code = href.strip("/").split("/")[-1].upper()
            
            if re.match(r"^[A-Z]{4}\d{4}$", code):
                rows.append({
                    "course_code": code,
                    "stream": stream_name,
                    "outline_url": full_url
                })
    
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=["course_code", "stream", "outline_url"]).reset_index(drop=True)
    return df

In [11]:
main_outline_df = get_outline_links(main_soup, main_url, "Common Year 1")
adm_outline_df = get_outline_links(adm_soup, adm_url, "ADM Option")

print("Common Year 1 outline links:", len(main_outline_df))
print("ADM outline links:", len(adm_outline_df))

main_outline_df.head(15)

Common Year 1 outline links: 14
,ADM outline links: 16


,course_code,stream,outline_url
0,BSYS1000,Common Year 1,https://www.bcit.ca/outlines/bsys1000/
1,COMM1100,Common Year 1,https://www.bcit.ca/outlines/comm1100/
2,ECON2100,Common Year 1,https://www.bcit.ca/outlines/econ2100/
3,MKTG1102,Common Year 1,https://www.bcit.ca/outlines/mktg1102/
4,OPMT1103,Common Year 1,https://www.bcit.ca/outlines/opmt1103/
5,OPMT1110,Common Year 1,https://www.bcit.ca/outlines/opmt1110/
6,BSYS2000,Common Year 1,https://www.bcit.ca/outlines/bsys2000/
7,BSYS2065,Common Year 1,https://www.bcit.ca/outlines/bsys2065/
8,BUSA2150,Common Year 1,https://www.bcit.ca/outlines/busa2150/
9,COMM2200,Common Year 1,https://www.bcit.ca/outlines/comm2200/


In [12]:
adm_outline_df.head(15)

,course_code,stream,outline_url
0,BABI3000,ADM Option,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,ADM Option,https://www.bcit.ca/outlines/babi3005/
2,BSYS3000,ADM Option,https://www.bcit.ca/outlines/bsys3000/
3,BSYS3205,ADM Option,https://www.bcit.ca/outlines/bsys3205/
4,BSYS3355,ADM Option,https://www.bcit.ca/outlines/bsys3355/
5,BUSA4850,ADM Option,https://www.bcit.ca/outlines/busa4850/
6,FMGT3221,ADM Option,https://www.bcit.ca/outlines/fmgt3221/
7,OPMT3301,ADM Option,https://www.bcit.ca/outlines/opmt3301/
8,BABI4000,ADM Option,https://www.bcit.ca/outlines/babi4000/
9,BABI4005,ADM Option,https://www.bcit.ca/outlines/babi4005/


## Scraping each course outline page

Now that I have the outline links, I can visit each course outline page and collect the actual course details. This should give me a cleaner result than trying to scrape everything from the main program pages.

In [13]:
def extract_course_name_from_page(soup, course_code):
    possible_texts = []
    
    h1 = soup.find("h1")
    if h1:
        possible_texts.append(h1.get_text(" ", strip=True))
    
    if soup.title:
        possible_texts.append(soup.title.get_text(" ", strip=True))
    
    for text in possible_texts:
        text = re.sub(r"\s+", " ", text).strip()
        text = text.replace(" - BCIT", "").replace("| BCIT", "").strip()
        
        pattern = rf"{course_code[:4]}\s?{course_code[4:]}\s*[-:]\s*(.+)"
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    
    if possible_texts:
        return possible_texts[0]
    
    return None

In [19]:
def extract_prerequisite_text(page_text):
    text = re.sub(r"\s+", " ", page_text).strip()
    
    patterns = [
        r"Prerequisite\(s\):\s*(.*?)(?=Corequisite\(s\):|Credits?:|Hours?:|Learning Outcomes|School:|Department:|$)",
        r"Prerequisite:\s*(.*?)(?=Corequisite:|Credits?:|Hours?:|Learning Outcomes|School:|Department:|$)",
        r"Pre/Corequisite\(s\):\s*(.*?)(?=Credits?:|Hours?:|Learning Outcomes|School:|Department:|$)"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            prereq = match.group(1).strip()
            if prereq:
                return prereq
    
    return None

In [15]:
def scrape_outline_details(row):
    url = row["outline_url"]
    course_code = row["course_code"]
    stream = row["stream"]
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")
        page_text = soup.get_text(" ", strip=True)
        
        course_name = extract_course_name_from_page(soup, course_code)
        prerequisite_text = extract_prerequisite_text(page_text)
        
        return {
            "course_code": course_code,
            "course_name": course_name,
            "stream": stream,
            "prerequisite_text": prerequisite_text,
            "source_url": url
        }
    
    except Exception as e:
        return {
            "course_code": course_code,
            "course_name": None,
            "stream": stream,
            "prerequisite_text": None,
            "source_url": url
        }

In [26]:
common_rows = []

for _, row in main_outline_df.iterrows():
    common_rows.append(scrape_outline_details(row))
    time.sleep(0.5)

common_courses_df = pd.DataFrame(common_rows)
common_courses_df

,course_code,course_name,stream,prerequisite_text,source_url
0,BSYS1000,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/bsys1000/
1,COMM1100,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/comm1100/
2,ECON2100,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/econ2100/
3,MKTG1102,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/mktg1102/
4,OPMT1103,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/opmt1103/
5,OPMT1110,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/opmt1110/
6,BSYS2000,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/bsys2000/
7,BSYS2065,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/bsys2065/
8,BUSA2150,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/busa2150/
9,COMM2200,Course Outline,Common Year 1,None,https://www.bcit.ca/outlines/comm2200/


In [17]:
adm_rows = []

for _, row in adm_outline_df.iterrows():
    adm_rows.append(scrape_outline_details(row))
    time.sleep(0.5)

adm_courses_df = pd.DataFrame(adm_rows)
adm_courses_df.head(15)

,course_code,course_name,stream,prerequisite_text,source_url
0,BABI3000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi3005/
2,BSYS3000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3000/
3,BSYS3205,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3205/
4,BSYS3355,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3355/
5,BUSA4850,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/busa4850/
6,FMGT3221,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/fmgt3221/
7,OPMT3301,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/opmt3301/
8,BABI4000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi4000/
9,BABI4005,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi4005/


In [18]:
final_df = pd.concat([common_courses_df, adm_courses_df], ignore_index=True)

final_df = final_df.drop_duplicates(subset=["course_code", "stream", "source_url"])
final_df = final_df.sort_values(by=["stream", "course_code"]).reset_index(drop=True)

print("Final rows:", len(final_df))
final_df

Final rows: 30


,course_code,course_name,stream,prerequisite_text,source_url
0,BABI3000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi3005/
2,BABI4000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi4000/
3,BABI4005,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/babi4005/
4,BLAW3600,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/blaw3600/
5,BSYS3000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3000/
6,BSYS3205,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3205/
7,BSYS3355,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys3355/
8,BSYS4000,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys4000/
9,BSYS4905,Course Outline,ADM Option,None,https://www.bcit.ca/outlines/bsys4905/


## Scraping the course pages

After testing different approaches, I found that the BCIT course pages were the best source for course names and prerequisites. So instead of using the generic outline pages for details, I use the course pages directly.


In [42]:
def extract_course_name_from_course_page(soup):
    h1 = soup.find("h1")
    if h1:
        text = h1.get_text(" ", strip=True)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    return None

In [43]:
def extract_prerequisite_text_from_course_page(soup):
    lines = [s.strip() for s in soup.stripped_strings]

    for i, line in enumerate(lines):
        if "Prerequisite" in line:
            prereq_parts = []
            j = i + 1

            while j < len(lines):
                next_line = lines[j]

                if next_line in ["Credits", "Learning Outcomes", "Related Programs", "Subscribe", "Contact"]:
                    break

                prereq_parts.append(next_line)
                j += 1

            prereq_text = " ".join(prereq_parts).strip()
            return prereq_text if prereq_text else None

    return None

In [45]:
def scrape_course_details(row):
    course_code = row["course_code"]
    stream = row["stream"]
    outline_url = row["outline_url"]

    course_page_url = f"https://www.bcit.ca/courses/{course_code.lower()}/"

    try:
        response = requests.get(course_page_url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        course_name = extract_course_name_from_course_page(soup)
        prerequisite_text = extract_prerequisite_text_from_course_page(soup)

        return {
            "course_code": course_code,
            "course_name": course_name,
            "stream": stream,
            "prerequisite_text": prerequisite_text,
            "source_url": course_page_url,
            "outline_url": outline_url
        }

    except Exception:
        return {
            "course_code": course_code,
            "course_name": None,
            "stream": stream,
            "prerequisite_text": None,
            "source_url": course_page_url,
            "outline_url": outline_url
        }

In [46]:
common_rows = []

for _, row in main_outline_df.iterrows():
    common_rows.append(scrape_course_details(row))
    time.sleep(0.5)

common_courses_df = pd.DataFrame(common_rows)
common_courses_df.head(15)

,course_code,course_name,stream,prerequisite_text,source_url,outline_url
0,BSYS1000,Business Information Systems BSYS 1000,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/bsys1000/,https://www.bcit.ca/outlines/bsys1000/
1,COMM1100,Business Communication 1 COMM 1100,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/comm1100/,https://www.bcit.ca/outlines/comm1100/
2,ECON2100,Microeconomics ECON 2100,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/econ2100/,https://www.bcit.ca/outlines/econ2100/
3,MKTG1102,Essentials of Marketing MKTG 1102,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/mktg1102/,https://www.bcit.ca/outlines/mktg1102/
4,OPMT1103,Introduction to Operations Management OPMT 1103,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/opmt1103/,https://www.bcit.ca/outlines/opmt1103/
5,OPMT1110,Business Mathematics OPMT 1110,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/opmt1110/,https://www.bcit.ca/outlines/opmt1110/
6,BSYS2000,Applied Data Analytics in Excel BSYS 2000,Common Year 1,50% in BSYS 1000 or 50% in BSYS 1001,https://www.bcit.ca/courses/bsys2000/,https://www.bcit.ca/outlines/bsys2000/
7,BSYS2065,Business Systems Programming BSYS 2065,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/bsys2065/,https://www.bcit.ca/outlines/bsys2065/
8,BUSA2150,Introduction to Organizational Development BUS...,Common Year 1,No prerequisites are required for this course.,https://www.bcit.ca/courses/busa2150/,https://www.bcit.ca/outlines/busa2150/
9,COMM2200,Business Communication 2 COMM 2200,Common Year 1,50% in COMM 1100,https://www.bcit.ca/courses/comm2200/,https://www.bcit.ca/outlines/comm2200/


In [47]:
adm_rows = []

for _, row in adm_outline_df.iterrows():
    adm_rows.append(scrape_course_details(row))
    time.sleep(0.5)

adm_courses_df = pd.DataFrame(adm_rows)
adm_courses_df.head(15)

,course_code,course_name,stream,prerequisite_text,source_url,outline_url
0,BABI3000,Data Quality Approaches for Business BABI 3000,ADM Option,50% in BSYS 2000 Successful completion of Year...,https://www.bcit.ca/courses/babi3000/,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,Data Governance for Business Analytics BABI 3005,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/babi3005/,https://www.bcit.ca/outlines/babi3005/
2,BSYS3000,Cloud Business Development 1 BSYS 3000,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3000/,https://www.bcit.ca/outlines/bsys3000/
3,BSYS3205,Business Intelligence 1 BSYS 3205,ADM Option,BSYS 2000,https://www.bcit.ca/courses/bsys3205/,https://www.bcit.ca/outlines/bsys3205/
4,BSYS3355,Management Information Systems BSYS 3355,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3355/,https://www.bcit.ca/outlines/bsys3355/
5,BUSA4850,Consulting Skills and Problem Solving BUSA 4850,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/busa4850/,https://www.bcit.ca/outlines/busa4850/
6,FMGT3221,Management Accounting Administration FMGT 3221,ADM Option,FMGT 2100 or FMGT 2152,https://www.bcit.ca/courses/fmgt3221/,https://www.bcit.ca/outlines/fmgt3221/
7,OPMT3301,Quantitative Methods for Business OPMT 3301,ADM Option,OPMT 1197 or an equivalent college level Busin...,https://www.bcit.ca/courses/opmt3301/,https://www.bcit.ca/outlines/opmt3301/
8,BABI4000,Applied Data Management for Analytics BABI 4000,ADM Option,50% in BABI 3000,https://www.bcit.ca/courses/babi4000/,https://www.bcit.ca/outlines/babi4000/
9,BABI4005,DQM with Python BABI 4005,ADM Option,50% in BABI 3005,https://www.bcit.ca/courses/babi4005/,https://www.bcit.ca/outlines/babi4005/


In [48]:
final_df = pd.concat([common_courses_df, adm_courses_df], ignore_index=True)

final_df = final_df.drop_duplicates(subset=["course_code", "stream", "source_url"])
final_df = final_df.sort_values(by=["stream", "course_code"]).reset_index(drop=True)

print("Final rows:", len(final_df))
final_df

Final rows: 30


,course_code,course_name,stream,prerequisite_text,source_url,outline_url
0,BABI3000,Data Quality Approaches for Business BABI 3000,ADM Option,50% in BSYS 2000 Successful completion of Year...,https://www.bcit.ca/courses/babi3000/,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,Data Governance for Business Analytics BABI 3005,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/babi3005/,https://www.bcit.ca/outlines/babi3005/
2,BABI4000,Applied Data Management for Analytics BABI 4000,ADM Option,50% in BABI 3000,https://www.bcit.ca/courses/babi4000/,https://www.bcit.ca/outlines/babi4000/
3,BABI4005,DQM with Python BABI 4005,ADM Option,50% in BABI 3005,https://www.bcit.ca/courses/babi4005/,https://www.bcit.ca/outlines/babi4005/
4,BLAW3600,Computers and the Law BLAW 3600,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/blaw3600/,https://www.bcit.ca/outlines/blaw3600/
5,BSYS3000,Cloud Business Development 1 BSYS 3000,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3000/,https://www.bcit.ca/outlines/bsys3000/
6,BSYS3205,Business Intelligence 1 BSYS 3205,ADM Option,BSYS 2000,https://www.bcit.ca/courses/bsys3205/,https://www.bcit.ca/outlines/bsys3205/
7,BSYS3355,Management Information Systems BSYS 3355,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3355/,https://www.bcit.ca/outlines/bsys3355/
8,BSYS4000,Cloud Business Development 2 BSYS 4000,ADM Option,BSYS 3000,https://www.bcit.ca/courses/bsys4000/,https://www.bcit.ca/outlines/bsys4000/
9,BSYS4905,Directed Studies BSYS 4905,ADM Option,"Successful completion of Levels 1, 2 and 3 cou...",https://www.bcit.ca/courses/bsys4905/,https://www.bcit.ca/outlines/bsys4905/


## Cleaning the course names

The course names were scraped correctly, but many of them still include the course code at the end. In this step, I am cleaning the course names so they look neater and are easier to read.

In [51]:
final_df["course_name"] = final_df.apply(
    lambda row: re.sub(
        rf"\s*{row['course_code'][:4]}\s?{row['course_code'][4:]}\s*$",
        "",
        row["course_name"]
    ).strip() if pd.notnull(row["course_name"]) else row["course_name"],
    axis=1
)

final_df.head(20)

,course_code,course_name,stream,prerequisite_text,source_url,outline_url
0,BABI3000,Data Quality Approaches for Business,ADM Option,50% in BSYS 2000 Successful completion of Year...,https://www.bcit.ca/courses/babi3000/,https://www.bcit.ca/outlines/babi3000/
1,BABI3005,Data Governance for Business Analytics,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/babi3005/,https://www.bcit.ca/outlines/babi3005/
2,BABI4000,Applied Data Management for Analytics,ADM Option,50% in BABI 3000,https://www.bcit.ca/courses/babi4000/,https://www.bcit.ca/outlines/babi4000/
3,BABI4005,DQM with Python,ADM Option,50% in BABI 3005,https://www.bcit.ca/courses/babi4005/,https://www.bcit.ca/outlines/babi4005/
4,BLAW3600,Computers and the Law,ADM Option,No prerequisites are required for this course.,https://www.bcit.ca/courses/blaw3600/,https://www.bcit.ca/outlines/blaw3600/
5,BSYS3000,Cloud Business Development 1,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3000/,https://www.bcit.ca/outlines/bsys3000/
6,BSYS3205,Business Intelligence 1,ADM Option,BSYS 2000,https://www.bcit.ca/courses/bsys3205/,https://www.bcit.ca/outlines/bsys3205/
7,BSYS3355,Management Information Systems,ADM Option,BSYS 2065,https://www.bcit.ca/courses/bsys3355/,https://www.bcit.ca/outlines/bsys3355/
8,BSYS4000,Cloud Business Development 2,ADM Option,BSYS 3000,https://www.bcit.ca/courses/bsys4000/,https://www.bcit.ca/outlines/bsys4000/
9,BSYS4905,Directed Studies,ADM Option,"Successful completion of Levels 1, 2 and 3 cou...",https://www.bcit.ca/courses/bsys4905/,https://www.bcit.ca/outlines/bsys4905/


In [53]:
final_df.to_csv("../data/bitman_adm_courses_cleaned.csv", index=False)


## What I found

This project worked better once I narrowed the scope and changed the scraping strategy. At first, I tried to pull everything from the program and outline pages, but that did not give me the actual details I needed. Using the course pages directly gave me cleaner course names and much better prerequisite information.

In the end, I built a structured dataset with 30 rows that covers Common Year 1 and the ADM option. This shows how webpage text can be turned into a dataset that is easier to read, compare, and use.
